In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy.stats import vonmises
from scipy.special import i0
from math import sin
from math import cos
import json
from IPython.display import display


# run functions

### `def wrap_angle_rad`

Wrap radian angles to $[-\pi, \pi)$ for circular-difference calculations.

In [ ]:
def wrap_angle_rad(theta):
    wrap = (theta + np.pi) % (2 * np.pi) - np.pi
    return wrap


### `def compute_log_likelihood`

Compute the log likelihood of one continuous trial sequence from the initial-state probabilities, transition matrix, and emission parameters.

In [ ]:
def compute_log_likelihood(
    data,
    params,
    transition_matrix,
    initial_probabilities,
):
    """
    Compute the log-likelihood of one continuous trial sequence.

    Parameters
    ----------
    data : dict
        Must contain:
        - x_rad
        - y_rad
        - coherence
        - prior_std

    params : dict
        Must contain:
        - kappaS: dict mapping coherence to sensory kappa
        - kappaP: dict mapping prior_std to prior kappa

    transition_matrix : array, shape (3, 3)
        Rows are previous states and columns are next states.

    initial_probabilities : array, shape (3,)
        Initial probabilities for Sensory, Prior, and Lapse states.

    Returns
    -------
    log_likelihood : float
        Log-likelihood of this one trial sequence.
    """

    x_rad = np.asarray(data["x_rad"], dtype=float)
    y_rad = np.asarray(data["y_rad"], dtype=float)
    coherence = np.asarray(data["coherence"])
    prior_std = np.asarray(data["prior_std"])

    transition_matrix = np.asarray(
        transition_matrix,
        dtype=float,
    )

    initial_probabilities = np.asarray(
        initial_probabilities,
        dtype=float,
    )

    n_trials = len(y_rad)

    if n_trials == 0:
        raise ValueError("The input data contains no trials.")

    if not (
        len(x_rad)
        == len(y_rad)
        == len(coherence)
        == len(prior_std)
    ):
        raise ValueError("All trial-level arrays must have the same length.")

    if transition_matrix.shape != (3, 3):
        raise ValueError("transition_matrix must have shape (3, 3).")

    if initial_probabilities.shape != (3,):
        raise ValueError("initial_probabilities must have shape (3,).")

    transition_matrix = (
        transition_matrix
        / transition_matrix.sum(axis=1, keepdims=True)
    )

    initial_probabilities = (
        initial_probabilities
        / initial_probabilities.sum()
    )

    emission_matrix = np.zeros((n_trials, 3))

    for t in range(n_trials):

        kappaS_t = params["kappaS"][coherence[t]]
        kappaP_t = params["kappaP"][prior_std[t]]

        sensory_likelihood = vonmises.pdf(
            y_rad[t],
            kappaS_t,
            loc=x_rad[t],
        )

        prior_likelihood = vonmises.pdf(
            y_rad[t],
            kappaP_t,
            loc=0.0,
        )

        lapse_likelihood = 1.0 / (2.0 * np.pi)

        emission_matrix[t] = [
            sensory_likelihood,
            prior_likelihood,
            lapse_likelihood,
        ]

    # Trial 1
    alpha = initial_probabilities * emission_matrix[0]

    scale = alpha.sum()

    if scale <= 0 or not np.isfinite(scale):
        return -np.inf

    alpha = alpha / scale
    log_likelihood = np.log(scale)

    # Trial 2 to Trial T
    for t in range(1, n_trials):

        predicted_probabilities = alpha @ transition_matrix

        alpha = (
            predicted_probabilities
            * emission_matrix[t]
        )

        scale = alpha.sum()

        if scale <= 0 or not np.isfinite(scale):
            return -np.inf

        alpha = alpha / scale
        log_likelihood += np.log(scale)

    return float(log_likelihood)

### `def prepare_hmm_data`

Convert motion directions and response coordinates to prior-centered radians and assemble the variables required by the HMM.

In [ ]:
def prepare_hmm_data(
        motion_direction,
        estimate_x,
        estimate_y,
        prior_mean,
        prior_std,
        motion_coherence,
        subject_id,
        run_id,
):
    esti_agl = np.atan2(estimate_y, estimate_x)

    motion_rad = np.deg2rad(motion_direction)
    prior_mean_rad = np.deg2rad(prior_mean)

    x_rad = wrap_angle_rad(motion_rad - prior_mean_rad)
    y_rad = wrap_angle_rad(esti_agl - prior_mean_rad)

    block_id = run_id

    data = {
        "x_rad": x_rad,
        "y_rad": y_rad,
        "coherence": motion_coherence,
        "prior_std": prior_std,
        "block_id": block_id,
        "subject_id": subject_id,
    }

    return data


## initialize functions

### `def initialize_parameters`

Initialize state probabilities, the transition matrix, and condition-specific concentration parameters for the soft EM-HMM.

In [ ]:
def initialize_parameters():
    params = {
        "initial_prob": np.array([0.60, 0.35, 0.05]),

        "transition_matrix": np.array([
            [0.80, 0.15, 0.05],
            [0.15, 0.80, 0.05],
            [0.40, 0.40, 0.20],
        ]),

        "kappaS": {
            0.06: 1.5,
            0.12: 4.5,
            0.24: 20.0,
        },

        "kappaP": {
            80: 0.5,
            40: 1.0,
            20: 6.0,
            10: 30.0,
        },
    }

    return params


## Emission function

### `def von_mises_density`

Evaluate the von Mises probability density for an angle, mean direction, and concentration parameter.

In [ ]:
def von_mises_density(theta, mu, kappa):
    p = np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * i0(kappa))
    return p


### `def get_sensory_kappa`

Return the sensory concentration parameter associated with the current coherence condition.

In [ ]:
def get_sensory_kappa(kappaS, coherence):
    return kappaS[coherence]


### `def get_prior_kappa`

Return the prior concentration parameter associated with the current prior-standard-deviation condition.

In [ ]:
def get_prior_kappa(kappaP, prior_std):
    return kappaP[prior_std]


### `def compute_trial_emission_likelihoods`

Compute the Sensory, Prior, and Lapse emission likelihoods for one trial.

In [ ]:
def compute_trial_emission_likelihoods(y_t, x_t, coherence, prior_std, params):
    bS = von_mises_density(
        y_t,
        x_t,
        get_sensory_kappa(params["kappaS"], coherence),
    )
    bP = von_mises_density(
        y_t,
        0.0,
        get_prior_kappa(params["kappaP"], prior_std),
    )
    bL = 1 / (2 * np.pi)

    return np.array([bS, bP, bL])


### `def compute_emission_matrix`

Apply the trial-level emission calculation to all trials and return an `n_trials × 3` emission matrix.

In [ ]:
def compute_emission_matrix(data, params):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["coherence"]
    prior_std = data["prior_std"]

    n_trials = len(x_rad)
    emission_matrix = np.zeros((n_trials, 3))

    for t in range(n_trials):
        emission_matrix[t] = compute_trial_emission_likelihoods(
            y_t=y_rad[t],
            x_t=x_rad[t],
            coherence=coherence[t],
            prior_std=prior_std[t],
            params=params,
        )

    return emission_matrix


## Forward-Backward function

### `def forward_pass`

Run the scaled forward recursion to obtain one-step-ahead state probabilities, filtered probabilities, scaling factors, and sequence log likelihood.

In [ ]:
def forward_pass(params, data):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["coherence"]
    prior_std = data["prior_std"]

    n = len(x_rad)
    n_states = 3

    A = params["transition_matrix"]
    initial_probs = params["initial_prob"]

    predicted_prob = np.zeros((n, n_states))
    filtered_prob = np.zeros((n, n_states))
    emission_likelihood = np.zeros((n, n_states))
    scales = np.zeros(n)

    log_likelihood = 0.0

    # Prior state probability before observing trial 0
    predicted_prob[0] = initial_probs

    for i in range(n):
        emission_likelihood[i] = compute_trial_emission_likelihoods(
            y_t=y_rad[i],
            x_t=x_rad[i],
            coherence=coherence[i],
            prior_std=prior_std[i],
            params=params,
        )

        filtered_unnormalized = (
            predicted_prob[i]
            * emission_likelihood[i]
        )

        scales[i] = filtered_unnormalized.sum()

        if scales[i] <= 0:
            raise ValueError(
                f"Forward scaling factor is zero at trial {i}."
            )

        filtered_prob[i] = (
            filtered_unnormalized / scales[i]
        )

        log_likelihood += np.log(scales[i])

        if i < n - 1:
            predicted_prob[i + 1] = (
                filtered_prob[i] @ A
            )

    return (
        predicted_prob,
        filtered_prob,
        emission_likelihood,
        scales,
        log_likelihood,
    )


### `def backward_pass`

Run the scaled backward recursion to quantify how future observations support each current hidden state.

In [ ]:
def backward_pass(params, data, scales, emission_likelihood):
    x_rad = data["x_rad"]

    n = len(x_rad)
    n_states = 3

    beta = np.zeros((n, n_states))
    A = params["transition_matrix"]

    beta[-1] = [1.0, 1.0, 1.0]

    for t in range(n - 2, -1, -1):
        future_support_next = (
            emission_likelihood[t + 1]
            * beta[t + 1]
        )

        beta[t] = A @ future_support_next
        beta[t] /= scales[t + 1]

    return beta

## M-step update

### `def update_state_posteriors`

Combine filtered probabilities with the backward messages to obtain the state posterior `gamma` for every trial.

In [ ]:
def update_state_posteriors(filtered_porb, beta):
    gamma_unnormalized = filtered_porb * beta
    gamma = gamma_unnormalized / gamma_unnormalized.sum(
        axis=1,
        keepdims=True
    )
    return gamma

### `def compute_transition_posteriors`

Compute `xi`, the joint posterior over each pair of adjacent hidden states.

In [ ]:
def compute_transition_posteriors(
        filtered_prob,
        beta,
        emission_likelihood,
        transition_matrix,
):
    next_state_support = (
        emission_likelihood[1:] * beta[1:]
    )

    xi_unnormalized = (
        filtered_prob[:-1, :, np.newaxis]
        * transition_matrix[np.newaxis, :, :]
        * next_state_support[:, np.newaxis, :]
    )

    normalizers = xi_unnormalized.sum(
        axis=(1, 2),
        keepdims=True,
    )

    if np.any(normalizers <= 0):
        raise ValueError(
            "Some transition posteriors cannot be normalized."
        )

    xi = xi_unnormalized / normalizers

    return xi

### `def compute_weighted_resultant_length`

Compute the weighted circular resultant length using state-posterior weights.

In [ ]:
def compute_weighted_resultant_length(error, weights):
    weighted_cos = np.sum(weights * np.cos(error))
    weighted_sin = np.sum(weights * np.sin(error))

    R = np.sqrt(
        weighted_cos**2 + weighted_sin**2
    ) / np.sum(weights)

    return R

### `def estimate_kappa_from_resultant`

Estimate a von Mises concentration parameter from the resultant length using the standard piecewise approximation.

In [ ]:
def estimate_kappa_from_resultant(R):
    R = np.clip(R, 0.0, 0.999999)

    if R < 0.53:
        kappa = 2 * R + R**3 + 5 * R**5 / 6

    elif R < 0.85:
        kappa = -0.4 + 1.39 * R + 0.43 / (1 - R)

    else:
        kappa = 1 / (R**3 - 4 * R**2 + 3 * R)

    return kappa

### `def update_sensory_kappas`

Update sensory concentration parameters by coherence using Sensory-state posterior weights.

In [ ]:
def update_sensory_kappas(data, gamma, params):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["coherence"]

    error_S = wrap_angle_rad(y_rad - x_rad)
    weights_S = gamma[:, 0]

    new_kappaS = {}

    for condition in params["kappaS"].keys():
        mask = coherence == condition

        condition_errors = error_S[mask]
        condition_weights = weights_S[mask]

        R = compute_weighted_resultant_length(
            condition_errors,
            condition_weights,
        )

        new_kappaS[condition] = (
            estimate_kappa_from_resultant(R)
        )

    return new_kappaS

### `def update_prior_kappas`

Update prior concentration parameters by prior width using Prior-state posterior weights.

In [ ]:
def update_prior_kappas(data, gamma, params):
    y_rad = data["y_rad"]
    prior_std = data["prior_std"]

    error_P = wrap_angle_rad(y_rad)
    weights_P = gamma[:, 1]

    new_kappaP = {}

    for condition in params["kappaP"].keys():
        mask = prior_std == condition

        condition_errors = error_P[mask]
        condition_weights = weights_P[mask]

        R = compute_weighted_resultant_length(
            condition_errors,
            condition_weights,
        )

        new_kappaP[condition] = (
            estimate_kappa_from_resultant(R)
        )

    return new_kappaP

### `def update_emission_parameters`

Update the sensory and prior emission parameters when `update_kappa` is enabled.

In [ ]:
def update_emission_parameters(data, gamma, params, update_kappa=True):
    if update_kappa:
        params["kappaS"] = update_sensory_kappas(
            data,
            gamma,
            params,
        )

        params["kappaP"] = update_prior_kappas(
            data,
            gamma,
            params,
        )

    return params

### `def update_transition_matrix`

Normalize posterior transition counts to obtain an updated transition matrix.

In [ ]:
def update_transition_matrix(xi, smoothing=1e-6):
    transition_counts = xi.sum(axis=0)
    transition_counts = transition_counts + smoothing

    transition_matrix = transition_counts / transition_counts.sum(
        axis=1,
        keepdims=True,
    )

    return transition_matrix


### `def update_initial_probabilities`

Average the first-trial state posteriors across independent blocks to update the initial-state probabilities.

In [ ]:
def update_initial_probabilities(block_initial_gammas, smoothing=1e-6):
    initial_counts = np.asarray(block_initial_gammas, dtype=float).sum(axis=0)
    initial_counts = initial_counts + smoothing
    return initial_counts / initial_counts.sum()


### `def m_step`

Perform one M-step, updating transitions and, when requested, the emission concentration parameters.

In [ ]:
def m_step(
        data,
        gamma,
        xi,
        params,
        update_kappa=True,
        update_pi=True,
        block_initial_gammas=None,
        transition_smoothing=1e-6,
):
    new_params = {
        "initial_prob": params["initial_prob"].copy(),
        "transition_matrix": params["transition_matrix"].copy(),
        "kappaS": params["kappaS"].copy(),
        "kappaP": params["kappaP"].copy(),
    }

    new_params["transition_matrix"] = update_transition_matrix(
        xi,
        smoothing=transition_smoothing,
    )

    if update_pi:
        new_params["initial_prob"] = update_initial_probabilities(
            block_initial_gammas,
            smoothing=transition_smoothing,
        )

    if update_kappa:
        new_params["kappaS"] = update_sensory_kappas(data, gamma, params)
        new_params["kappaP"] = update_prior_kappas(data, gamma, params)

    return new_params


# full-training function

### `def compute_total_log_likelihood`

Sum block-level log likelihoods to obtain the total observed-data log likelihood.

In [ ]:
def compute_total_log_likelihood(block_log_likelihoods):
    total_log_likelihood = np.sum(block_log_likelihoods)
    return float(total_log_likelihood)


### `def check_convergence`

Compare successive total log likelihoods against the stopping tolerance.

In [ ]:
def check_convergence(
        current_log_likelihood,
        previous_log_likelihood,
        tol=1e-6,
):
    if previous_log_likelihood is None:
        return False

    difference = abs(
        current_log_likelihood - previous_log_likelihood
    )

    return difference < tol


### `def fit_soft_em_hmm`

Alternate block-wise forward-backward inference and M-steps until convergence or the iteration limit.

In [ ]:
def fit_soft_em_hmm(
        data,
        params=None,
        update_kappa=True,
        update_pi=True,
        transition_smoothing=1e-6,
        max_iter=100,
        tol=1e-6,
):
    if params is None:
        params = initialize_parameters()

    current_params = {
        "initial_prob": params["initial_prob"].copy(),
        "transition_matrix": params["transition_matrix"].copy(),
        "kappaS": params["kappaS"].copy(),
        "kappaP": params["kappaP"].copy(),
    }

    block_ids = np.asarray(data["block_id"])
    unique_blocks = pd.unique(block_ids)
    log_likelihood_history = []
    previous_log_likelihood = None
    converged = False
    final_gamma = None
    final_xi = None

    for iteration in range(max_iter):
        gamma_list = []
        xi_list = []
        block_initial_gammas = []
        block_log_likelihoods = []
        combined_data = {"x_rad": [], "y_rad": [], "coherence": [], "prior_std": []}

        for block in unique_blocks:
            mask = block_ids == block
            block_data = {
                "x_rad": np.asarray(data["x_rad"])[mask],
                "y_rad": np.asarray(data["y_rad"])[mask],
                "coherence": np.asarray(data["coherence"])[mask],
                "prior_std": np.asarray(data["prior_std"])[mask],
            }
            (
                predicted_prob,
                filtered_prob,
                emission_likelihood,
                scales,
                block_log_likelihood,
            ) = forward_pass(current_params, block_data)
            beta = backward_pass(current_params, block_data, scales, emission_likelihood)
            gamma = update_state_posteriors(filtered_prob, beta)
            xi = compute_transition_posteriors(
                filtered_prob,
                beta,
                emission_likelihood,
                current_params["transition_matrix"],
            )
            gamma_list.append(gamma)
            xi_list.append(xi)
            block_initial_gammas.append(gamma[0])
            block_log_likelihoods.append(block_log_likelihood)
            for key in combined_data:
                combined_data[key].append(block_data[key])

        gamma_all = np.concatenate(gamma_list, axis=0)
        xi_all = np.concatenate(xi_list, axis=0)
        combined_data = {
            key: np.concatenate(value, axis=0)
            for key, value in combined_data.items()
        }
        total_log_likelihood = compute_total_log_likelihood(block_log_likelihoods)
        log_likelihood_history.append(total_log_likelihood)
        converged = check_convergence(total_log_likelihood, previous_log_likelihood, tol)
        final_gamma = gamma_all
        final_xi = xi_all

        if converged or iteration == max_iter - 1:
            break

        current_params = m_step(
            combined_data,
            gamma_all,
            xi_all,
            current_params,
            update_kappa=update_kappa,
            update_pi=update_pi,
            block_initial_gammas=block_initial_gammas,
            transition_smoothing=transition_smoothing,
        )
        previous_log_likelihood = total_log_likelihood

    return {
        "params": current_params,
        "gamma": final_gamma,
        "xi": final_xi,
        "log_likelihood_history": log_likelihood_history,
        "converged": converged,
        "n_iterations": len(log_likelihood_history),
    }


# Viterbi-decoding function

### `def viterbi_decode_block`

Decode the most likely hidden-state path for one continuous block using fitted parameters.

In [ ]:
def viterbi_decode_block(data, params):
    emission_matrix = compute_emission_matrix(data, params)

    n_trials = len(data["x_rad"])
    n_states = 3

    log_initial = np.log(params["initial_prob"])
    log_transition = np.log(params["transition_matrix"])
    log_emission = np.log(emission_matrix)

    delta = np.zeros((n_trials, n_states))
    psi = np.zeros((n_trials, n_states), dtype=int)

    delta[0] = log_initial + log_emission[0]

    for t in range(1, n_trials):
        for current_state in range(n_states):
            previous_scores = (
                delta[t - 1]
                + log_transition[:, current_state]
            )

            psi[t, current_state] = np.argmax(previous_scores)
            delta[t, current_state] = (
                previous_scores[psi[t, current_state]]
                + log_emission[t, current_state]
            )

    state_path = np.zeros(n_trials, dtype=int)
    state_path[-1] = np.argmax(delta[-1])

    for t in range(n_trials - 2, -1, -1):
        state_path[t] = psi[t + 1, state_path[t + 1]]

    return state_path


### `def viterbi_decode_all_blocks`

Run Viterbi decoding independently within each block and return states in the original trial order.

In [ ]:
def viterbi_decode_all_blocks(data, params):
    block_ids = np.asarray(data["block_id"])
    unique_blocks = pd.unique(block_ids)

    all_state_paths = np.zeros(len(block_ids), dtype=int)

    for block in unique_blocks:
        mask = block_ids == block

        block_data = {
            "x_rad": np.asarray(data["x_rad"])[mask],
            "y_rad": np.asarray(data["y_rad"])[mask],
            "coherence": np.asarray(data["coherence"])[mask],
            "prior_std": np.asarray(data["prior_std"])[mask],
        }

        block_state_path = viterbi_decode_block(
            block_data,
            params,
        )

        all_state_paths[mask] = block_state_path

    return all_state_paths


# Independent soft EM-HMM fits for all participants

Apply the validated Subject 1 functions and common training settings independently to every participant. A failed participant does not interrupt the remaining fits.

## Cell A: Common training settings

All participants use the same validated settings. Quality flags identify state collapse, near-zero lapse weight, and transition probabilities at the smoothing boundary.

In [ ]:
max_iter = 300
tol = 1e-6
transition_smoothing = 1e-6
update_pi = True
update_kappa = True
random_seed = 20260717
np.random.seed(random_seed)

state_collapse_threshold = 0.98
lapse_near_zero_threshold = 0.005
transition_near_floor_threshold = 1e-6

training_settings = {
    "max_iter": max_iter,
    "tol": tol,
    "transition_smoothing": transition_smoothing,
    "update_pi": update_pi,
    "update_kappa": update_kappa,
    "random_seed": random_seed,
    "initial_pi": [0.60, 0.35, 0.05],
    "initial_transition_matrix": [
        [0.80, 0.15, 0.05],
        [0.15, 0.80, 0.05],
        [0.40, 0.40, 0.20],
    ],
    "initial_sensory_kappas": {"0.06": 1.5, "0.12": 4.5, "0.24": 20.0},
    "initial_prior_kappas": {"80": 0.5, "40": 1.0, "20": 6.0, "10": 30.0},
    "state_collapse_threshold": state_collapse_threshold,
    "lapse_near_zero_threshold": lapse_near_zero_threshold,
    "transition_near_floor_threshold": transition_near_floor_threshold,
}

output_root = Path("all_subject_results")
output_root.mkdir(exist_ok=True)


## Cell B: Load and validate the full dataset

Load the CSV, verify required fields, and report participant, trial, and block counts without adding exploratory analyses.

In [ ]:
csv_path = Path("data01_direction4priors.csv")
prepared_data = pd.read_csv(csv_path)
required_columns = [
    "motion_direction", "motion_coherence", "estimate_x", "estimate_y",
    "prior_mean", "prior_std", "subject_id", "session_id", "run_id",
    "trial_index",
]
missing_columns = [column for column in required_columns if column not in prepared_data.columns]
if missing_columns:
    raise KeyError(f"CSV is missing required columns: {missing_columns}")

subject_ids = sorted(prepared_data["subject_id"].unique())
input_overview = prepared_data.groupby("subject_id").agg(
    n_trials=("trial_index", "size"),
    n_sessions=("session_id", "nunique"),
)
input_overview["n_blocks"] = prepared_data.groupby(
    ["subject_id", "session_id", "run_id"]
).size().groupby("subject_id").size()
input_overview["missing_required_values"] = (
    prepared_data[required_columns].isna()
    .groupby(prepared_data["subject_id"])
    .sum()
    .sum(axis=1)
    .astype(int)
)
print(f"Number of participants: {len(subject_ids)}")
display(input_overview)


## Cell C: Participant-level execution function

Select one participant, construct independent blocks, fit the existing HMM, run numerical checks and Viterbi decoding, save the outputs, and return a participant-level summary.

In [ ]:
def run_single_subject(subject_id, prepared_data, training_settings, output_root):
    stage = "data_selection"
    try:
        subject_id = int(subject_id)
        subject_df = (
            prepared_data.loc[prepared_data["subject_id"] == subject_id]
            .sort_values(["subject_id", "session_id", "run_id", "trial_index"], kind="stable")
            .reset_index(drop=True)
        )
        if subject_df.empty:
            raise ValueError(f"Subject {subject_id} has no valid trials.")

        stage = "data_validation"
        model_columns = [
            "motion_direction", "motion_coherence", "estimate_x", "estimate_y",
            "prior_mean", "prior_std", "subject_id", "session_id", "run_id",
            "trial_index",
        ]
        missing_counts = subject_df[model_columns].isna().sum()
        if int(missing_counts.sum()) > 0:
            missing_detail = missing_counts[missing_counts > 0].to_dict()
            raise ValueError(f"Required model fields contain missing values: {missing_detail}")

        block_sizes = subject_df.groupby(["session_id", "run_id"], sort=False).size()
        if len(block_sizes) == 0 or int(block_sizes.min()) < 1:
            raise ValueError("An empty block was detected.")
        trial_order_ok = subject_df.groupby(
            ["session_id", "run_id"], sort=False
        )["trial_index"].apply(lambda values: values.is_monotonic_increasing).all()
        if not trial_order_ok:
            raise ValueError("`trial_index` is not increasing within a block.")

        stage = "block_construction"
        block_labels = subject_df["session_id"].astype(str).str.cat(
            subject_df["run_id"].astype(str), sep="_"
        ).to_numpy()
        subject_data = prepare_hmm_data(
            motion_direction=subject_df["motion_direction"].to_numpy(),
            estimate_x=subject_df["estimate_x"].to_numpy(),
            estimate_y=subject_df["estimate_y"].to_numpy(),
            prior_mean=subject_df["prior_mean"].to_numpy(),
            prior_std=subject_df["prior_std"].to_numpy(),
            motion_coherence=subject_df["motion_coherence"].to_numpy(),
            subject_id=subject_df["subject_id"].to_numpy(),
            run_id=block_labels,
        )
        n_blocks = len(pd.unique(subject_data["block_id"]))
        if n_blocks != len(block_sizes):
            raise AssertionError("Constructed block count does not match the `session_id + run_id` grouping.")

        initial_params = {
            "initial_prob": np.asarray(training_settings["initial_pi"], dtype=float),
            "transition_matrix": np.asarray(
                training_settings["initial_transition_matrix"], dtype=float
            ),
            "kappaS": {
                float(key): float(value)
                for key, value in training_settings["initial_sensory_kappas"].items()
            },
            "kappaP": {
                int(key): float(value)
                for key, value in training_settings["initial_prior_kappas"].items()
            },
        }

        stage = "soft_em_fit"
        fit_results = fit_soft_em_hmm(
            data=subject_data,
            params=initial_params,
            update_kappa=training_settings["update_kappa"],
            update_pi=training_settings["update_pi"],
            transition_smoothing=training_settings["transition_smoothing"],
            max_iter=training_settings["max_iter"],
            tol=training_settings["tol"],
        )
        final_params = fit_results["params"]
        gamma = np.asarray(fit_results["gamma"], dtype=float)
        ll_history = np.asarray(fit_results["log_likelihood_history"], dtype=float)
        transition_matrix = np.asarray(final_params["transition_matrix"], dtype=float)
        kappa_values = np.asarray(
            list(final_params["kappaS"].values()) + list(final_params["kappaP"].values()),
            dtype=float,
        )

        stage = "numeric_validation"
        arrays_for_checks = [gamma, ll_history, transition_matrix, kappa_values]
        has_nan = bool(any(np.isnan(values).any() for values in arrays_for_checks))
        has_inf = bool(any(np.isinf(values).any() for values in arrays_for_checks))
        numeric_checks = {
            "gamma_rows_sum_to_one": np.allclose(gamma.sum(axis=1), 1.0, atol=1e-8),
            "transition_rows_sum_to_one": np.allclose(transition_matrix.sum(axis=1), 1.0, atol=1e-10),
            "gamma_in_unit_interval": np.all((gamma >= 0.0) & (gamma <= 1.0)),
            "transition_in_unit_interval": np.all((transition_matrix >= 0.0) & (transition_matrix <= 1.0)),
            "all_kappas_positive": np.all(kappa_values > 0.0),
            "no_nan": not has_nan,
            "no_inf": not has_inf,
        }
        if not all(bool(value) for value in numeric_checks.values()):
            failed_checks = [name for name, passed in numeric_checks.items() if not passed]
            raise AssertionError(f"Numerical checks failed: {failed_checks}")

        stage = "viterbi_decoding"
        state_names = np.array(["sensory", "prior", "lapse"])
        viterbi_states = viterbi_decode_all_blocks(subject_data, final_params)
        state_weights = gamma.mean(axis=0)
        minimum_transition = float(transition_matrix.min())
        maximum_transition = float(transition_matrix.max())

        stage = "result_saving"
        subject_output = output_root / f"subject_{subject_id:02d}"
        subject_output.mkdir(parents=True, exist_ok=True)
        trial_results = subject_df.copy()
        trial_results["block_id"] = subject_data["block_id"]
        trial_results["x_rad"] = subject_data["x_rad"]
        trial_results["y_rad"] = subject_data["y_rad"]
        trial_results["coherence"] = subject_data["coherence"]
        trial_results["p_sensory"] = gamma[:, 0]
        trial_results["p_prior"] = gamma[:, 1]
        trial_results["p_lapse"] = gamma[:, 2]
        trial_results["most_likely_state"] = state_names[np.argmax(gamma, axis=1)]
        trial_results["viterbi_state"] = state_names[viterbi_states]
        trial_output_columns = [
            "subject_id", "session_id", "run_id", "block_id", "trial_index",
            "x_rad", "y_rad", "coherence", "prior_std", "p_sensory",
            "p_prior", "p_lapse", "most_likely_state", "viterbi_state",
        ]
        trial_results[trial_output_columns].to_csv(
            subject_output / f"subject_{subject_id:02d}_trial_posteriors.csv",
            index=False,
        )

        likelihood_table = pd.DataFrame({
            "iteration": np.arange(1, len(ll_history) + 1),
            "log_likelihood": ll_history,
        })
        likelihood_table.to_csv(
            subject_output / f"subject_{subject_id:02d}_log_likelihood_history.csv",
            index=False,
        )

        summary = {
            "subject_id": subject_id,
            "success": True,
            "n_trials": int(len(subject_df)),
            "n_blocks": int(n_blocks),
            "converged": bool(fit_results["converged"]),
            "n_iterations": int(fit_results["n_iterations"]),
            "initial_log_likelihood": float(ll_history[0]),
            "final_log_likelihood": float(ll_history[-1]),
            "log_likelihood_improvement": float(ll_history[-1] - ll_history[0]),
            "sensory_weight": float(state_weights[0]),
            "prior_weight": float(state_weights[1]),
            "lapse_weight": float(state_weights[2]),
            "min_transition_probability": minimum_transition,
            "max_transition_probability": maximum_transition,
            "kappa_s_6": float(final_params["kappaS"][0.06]),
            "kappa_s_12": float(final_params["kappaS"][0.12]),
            "kappa_s_24": float(final_params["kappaS"][0.24]),
            "kappa_p_10": float(final_params["kappaP"][10]),
            "kappa_p_20": float(final_params["kappaP"][20]),
            "kappa_p_40": float(final_params["kappaP"][40]),
            "kappa_p_80": float(final_params["kappaP"][80]),
            "has_nan": has_nan,
            "has_inf": has_inf,
            "reached_max_iter": bool(
                not fit_results["converged"]
                and fit_results["n_iterations"] >= training_settings["max_iter"]
            ),
            "state_collapse_flag": bool(np.any(state_weights > training_settings["state_collapse_threshold"])),
            "lapse_near_zero_flag": bool(state_weights[2] < training_settings["lapse_near_zero_threshold"]),
            "transition_near_smoothing_floor_flag": bool(
                minimum_transition <= training_settings["transition_near_floor_threshold"]
            ),
        }

        parameter_payload = {
            "subject_id": subject_id,
            "n_trials": summary["n_trials"],
            "n_blocks": summary["n_blocks"],
            "converged": summary["converged"],
            "n_iterations": summary["n_iterations"],
            "initial_log_likelihood": summary["initial_log_likelihood"],
            "final_log_likelihood": summary["final_log_likelihood"],
            "log_likelihood_improvement": summary["log_likelihood_improvement"],
            "pi": np.asarray(final_params["initial_prob"], dtype=float).tolist(),
            "transition_matrix": transition_matrix.tolist(),
            "sensory_kappas": {
                str(key): float(value) for key, value in final_params["kappaS"].items()
            },
            "prior_kappas": {
                str(key): float(value) for key, value in final_params["kappaP"].items()
            },
            "sensory_weight": summary["sensory_weight"],
            "prior_weight": summary["prior_weight"],
            "lapse_weight": summary["lapse_weight"],
            "training_settings": training_settings,
            "minimum_transition_probability": minimum_transition,
            "maximum_transition_probability": maximum_transition,
        }
        with (subject_output / f"subject_{subject_id:02d}_parameters.json").open(
            "w", encoding="utf-8"
        ) as handle:
            json.dump(parameter_payload, handle, ensure_ascii=False, indent=2)

        transition_record = {"subject_id": subject_id}
        for previous_index, previous_name in enumerate(state_names):
            for next_index, next_name in enumerate(state_names):
                transition_record[f"{previous_name}_to_{next_name}"] = float(
                    transition_matrix[previous_index, next_index]
                )
        summary["transition_record"] = transition_record
        return summary

    except Exception as error:
        error.subject_stage = stage
        raise


## Cell D: Fit every participant

Run participants in sorted order. Failures are recorded without stopping later fits, and unavailable metrics remain missing in the summary.

In [ ]:
summary_records = []
failed_records = []
transition_records = []

summary_columns = [
    "subject_id", "success", "n_trials", "n_blocks", "converged",
    "n_iterations", "initial_log_likelihood", "final_log_likelihood",
    "log_likelihood_improvement", "sensory_weight", "prior_weight",
    "lapse_weight", "min_transition_probability", "max_transition_probability",
    "kappa_s_6", "kappa_s_12", "kappa_s_24", "kappa_p_10",
    "kappa_p_20", "kappa_p_40", "kappa_p_80", "has_nan", "has_inf",
    "reached_max_iter", "state_collapse_flag", "lapse_near_zero_flag",
    "transition_near_smoothing_floor_flag",
]

for current_subject_id in subject_ids:
    try:
        summary = run_single_subject(
            subject_id=current_subject_id,
            prepared_data=prepared_data,
            training_settings=training_settings,
            output_root=output_root,
        )
        transition_records.append(summary.pop("transition_record"))
        summary_records.append(summary)
        print(
            f"Subject {int(current_subject_id):02d}: success; "
            f"converged={summary['converged']}; iterations={summary['n_iterations']}"
        )
    except Exception as error:
        current_subject_id = int(current_subject_id)
        subject_frame = prepared_data.loc[prepared_data["subject_id"] == current_subject_id]
        failed_records.append({
            "subject_id": current_subject_id,
            "error_type": type(error).__name__,
            "error_message": str(error),
            "stage": getattr(error, "subject_stage", "unknown"),
        })
        failed_summary = {column: np.nan for column in summary_columns}
        failed_summary.update({
            "subject_id": current_subject_id,
            "success": False,
            "n_trials": int(len(subject_frame)),
            "n_blocks": int(subject_frame.groupby(["session_id", "run_id"]).ngroups),
            "converged": False,
            "reached_max_iter": False,
        })
        summary_records.append(failed_summary)
        print(
            f"Subject {current_subject_id:02d}: failed at "
            f"{getattr(error, 'subject_stage', 'unknown')} - {type(error).__name__}: {error}"
        )

all_subject_summary = pd.DataFrame(summary_records).reindex(columns=summary_columns)
all_subject_summary = all_subject_summary.sort_values("subject_id").reset_index(drop=True)
failed_subjects = pd.DataFrame(
    failed_records,
    columns=["subject_id", "error_type", "error_message", "stage"],
)
transition_table = pd.DataFrame(transition_records).sort_values("subject_id").reset_index(drop=True)

all_subject_summary.to_csv(output_root / "all_subject_summary.csv", index=False)
failed_subjects.to_csv(output_root / "failed_subjects.csv", index=False)
transition_table.to_csv(output_root / "all_subject_transition_matrices.csv", index=False)


## Cell E: Minimum batch-output checks

Check success, failure, convergence, quality flags, and the required output files for every successful participant.

In [ ]:
successful_summary = all_subject_summary.loc[all_subject_summary["success"] == True].copy()
successful_subject_ids = successful_summary["subject_id"].astype(int).tolist()
failed_subject_ids = failed_subjects["subject_id"].astype(int).tolist()

missing_output_files = []
for successful_subject_id in successful_subject_ids:
    subject_directory = output_root / f"subject_{successful_subject_id:02d}"
    expected_files = [
        subject_directory / f"subject_{successful_subject_id:02d}_parameters.json",
        subject_directory / f"subject_{successful_subject_id:02d}_trial_posteriors.csv",
        subject_directory / f"subject_{successful_subject_id:02d}_log_likelihood_history.csv",
    ]
    missing_output_files.extend(str(path) for path in expected_files if not path.exists())
if missing_output_files:
    raise AssertionError(f"Successful participants are missing output files: {missing_output_files}")

quality_report = {
    "total_subjects": len(subject_ids),
    "successful_subjects": len(successful_subject_ids),
    "failed_subjects": len(failed_subject_ids),
    "converged_subjects": int(successful_summary["converged"].fillna(False).sum()),
    "nonconverged_subjects": int((~successful_summary["converged"].fillna(False).astype(bool)).sum()),
    "reached_max_iter_subjects": successful_summary.loc[
        successful_summary["reached_max_iter"].fillna(False), "subject_id"
    ].astype(int).tolist(),
    "state_collapse_subjects": successful_summary.loc[
        successful_summary["state_collapse_flag"].fillna(False), "subject_id"
    ].astype(int).tolist(),
    "lapse_near_zero_subjects": successful_summary.loc[
        successful_summary["lapse_near_zero_flag"].fillna(False), "subject_id"
    ].astype(int).tolist(),
    "transition_near_floor_subjects": successful_summary.loc[
        successful_summary["transition_near_smoothing_floor_flag"].fillna(False), "subject_id"
    ].astype(int).tolist(),
    "nan_or_inf_subjects": successful_summary.loc[
        successful_summary["has_nan"].fillna(False)
        | successful_summary["has_inf"].fillna(False), "subject_id"
    ].astype(int).tolist(),
    "failed_subject_ids": failed_subject_ids,
    "all_success_outputs_exist": len(missing_output_files) == 0,
}
display(pd.Series(quality_report, name="value").to_frame())
display(all_subject_summary)
display(failed_subjects)


## Cell F: Essential all-participant quality-control figures

Plot mean state posteriors, final log likelihoods, and iteration counts. Participant transition matrices are saved separately.

In [ ]:
plot_summary = successful_summary.sort_values("subject_id")
plot_subjects = plot_summary["subject_id"].astype(int).to_numpy()
x_positions = np.arange(len(plot_subjects))
bar_width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(x_positions - bar_width, plot_summary["sensory_weight"], bar_width, label="Sensory")
axes[0].bar(x_positions, plot_summary["prior_weight"], bar_width, label="Prior")
axes[0].bar(x_positions + bar_width, plot_summary["lapse_weight"], bar_width, label="Lapse")
axes[0].set_xticks(x_positions, plot_subjects)
axes[0].set(xlabel="Subject", ylabel="Mean posterior", title="State weights by subject", ylim=(0, 1))
axes[0].legend()

axes[1].bar(plot_subjects.astype(str), plot_summary["final_log_likelihood"], color="#4C78A8")
axes[1].set(xlabel="Subject", ylabel="Final log-likelihood", title="Final log-likelihood")

axes[2].bar(plot_subjects.astype(str), plot_summary["n_iterations"], color="#F58518")
axes[2].axhline(max_iter, color="black", linestyle="--", linewidth=1, label="max_iter")
axes[2].set(xlabel="Subject", ylabel="Iterations", title="Iteration count")
axes[2].legend()

fig.tight_layout()
plt.show()
